# hvad-skal-vi-seerr - training notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shuresm57/hvad-skal-vi-seerr/blob/main/notebooks/train.ipynb)

Trains a small classifier on **your own** Jellyfin watch history + Jellyseerr/Overseerr
request history, to rank the unwatched titles already sitting in your library.

1. On a machine that can reach your media server, run `export_history.py` locally
   (see the repo README) to produce `library.csv`.
2. Run every cell in this notebook top to bottom (`Runtime > Run all`) and upload
   `library.csv` when the next cell asks for it.
3. The last cell downloads `models.zip`. Unzip it into `models/` in your local clone,
   then run locally:

```
pip install -r requirements/infer.txt
python recommend.py --library library.csv
```

`library.csv` contains your personal viewing data - it's only uploaded to this Colab
session, never committed to the repo.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()  # select the library.csv you exported locally
    csv_path = next(iter(uploaded))
except ImportError:
    csv_path = 'library.csv'

print('using', csv_path)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

df = pd.read_csv(csv_path)
df['genres'] = df['genres'].fillna('').apply(lambda g: g.split('|') if g else [])

genre_vocab = sorted({g for genres in df['genres'] for g in genres})
genre_index = {g: i for i, g in enumerate(genre_vocab)}

def genre_vector(genres):
    vec = np.zeros(len(genre_vocab), dtype=np.float32)
    for g in genres:
        vec[genre_index[g]] = 1.0
    return vec

genre_features = np.stack(df['genres'].apply(genre_vector))

year = df['year'].fillna(df['year'].median())
year_mean, year_std = float(year.mean()), float(year.std() or 1.0)
year_feature = ((year - year_mean) / year_std).to_numpy(dtype=np.float32).reshape(-1, 1)

X = np.concatenate([genre_features, year_feature], axis=1)
y = df['label'].to_numpy(dtype=np.float32)
n_features = X.shape[1]

print(f'{len(df)} items, {len(genre_vocab)} genres, {int(y.sum())} positive / {int(len(y) - y.sum())} negative')

In [ ]:
perm = rng.permutation(len(df))
split = int(0.8 * len(df))
train_idx, val_idx = perm[:split], perm[split:]

X_train = torch.tensor(X[train_idx])
y_train = torch.tensor(y[train_idx])
X_val = torch.tensor(X[val_idx])
y_val = torch.tensor(y[val_idx])

n_pos = y_train.sum().clamp(min=1)
n_neg = (len(y_train) - y_train.sum()).clamp(min=1)
pos_weight = n_neg / n_pos

model = nn.Linear(n_features, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=1e-3)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
EPOCHS = 200

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train).squeeze(1)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0 or epoch == 1:
        print(f'epoch {epoch:>3}  train_loss={loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(X_val).squeeze(1)
    val_pred = torch.sigmoid(val_logits)
    val_acc = ((val_pred > 0.5).float() == y_val).float().mean().item()

    order = torch.argsort(val_logits, descending=True)
    ranked_labels = y_val[order]
    k = min(10, len(ranked_labels))
    precision_at_k = ranked_labels[:k].mean().item() if k else float('nan')

print(f'val accuracy: {val_acc:.3f}')
print(f'val precision@{k}: {precision_at_k:.3f}')

In [ ]:
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

weight = model.weight.detach().numpy().reshape(-1)
bias = float(model.bias.detach().item())

np.save(models_dir / 'weights.npy', weight)
(models_dir / 'bias.json').write_text(json.dumps({'bias': bias}))
(models_dir / 'genre_vocab.json').write_text(json.dumps(genre_vocab))
(models_dir / 'year_stats.json').write_text(json.dumps({'mean': year_mean, 'std': year_std}))
(models_dir / 'metrics.json').write_text(json.dumps({
    'val_accuracy': val_acc,
    f'val_precision_at_{k}': precision_at_k,
    'n_items': len(df),
    'n_genres': len(genre_vocab),
    'n_positive': int(y.sum()),
}, indent=2))

print('artifacts written to', models_dir.resolve())
print(sorted(p.name for p in models_dir.iterdir()))

In [ ]:
import shutil

shutil.make_archive('models', 'zip', root_dir='.', base_dir='models')

try:
    from google.colab import files
    files.download('models.zip')
except ImportError:
    print('Not running in Colab - models.zip was created in the current directory, copy it manually.')

## What gets exported

- `weights.npy`, `bias.json` - the trained linear classifier
- `genre_vocab.json`, `year_stats.json` - feature encoding needed to score new items the same way
- `metrics.json` - validation accuracy/precision and dataset size, so you have a record of what you actually trained

## Note on the model

This is a single-user content-based model (genres + release year -> did-you-watch-it),
not collaborative filtering - with only one user's history, there's no "other users"
to collaborate with. It's deliberately simple: a linear classifier is enough to show
the train/serve MLOps split without needing a bigger dataset or heavier infra.